# Fine Tuning with Enterprise Model

Personal note (please refer to this): 

https://claude.ai/chat/0e0038df-16b7-418e-90fc-ef4969f8e769

In [1]:
import openai
import json
import os
import time
from dotenv import load_dotenv

load_dotenv()

# Set your API key
api_key = os.getenv("OPENAI_API_KEY")
if not api_key:
    raise ValueError("OPENAI_API_KEY not found in environment variables. Please set it in your .env file.")
openai.api_key = api_key

## Step 1: Create your training data file

In [4]:
# Step 1: Create your training data file
with open('training_data.json', 'r') as f:
    training_data = json.load(f)

# Save to JSONL file
with open('training_data.jsonl', 'w') as f:
    for item in training_data:
        f.write(json.dumps(item) + '\n')

print("Training data file created: training_data.jsonl")

Training data file created: training_data.jsonl


## Step 2: Upload the training file

In [2]:
# Step 2: Upload the training file
print("\nUploading training file...")
with open('training_data.jsonl', 'rb') as f:
    response = openai.files.create(
        file=f,
        purpose='fine-tune'
    )

file_id = response.id
print(f"File uploaded successfully. File ID: {file_id}")


Uploading training file...
File uploaded successfully. File ID: file-SKEz1UwLHqodnnvbeoeMuj


## Step 3: Create fine-tuning job

In [3]:
# Step 3: Create fine-tuning job
print("\nStarting fine-tuning job...")
fine_tune_response = openai.fine_tuning.jobs.create(
    training_file=file_id,
    model="gpt-4o-mini-2024-07-18",
    hyperparameters={
        "n_epochs": 3  # Number of training cycles (1-50, default is auto)
    }
)

job_id = fine_tune_response.id
print(f"Fine-tuning job created. Job ID: {job_id}")


Starting fine-tuning job...
Fine-tuning job created. Job ID: ftjob-oSD6lJDqPIsw3qGcgyrOm6WD


## Step 4: Monitor the fine-tuning progress

In [ ]:
# Step 4: Monitor the fine-tuning progress
print("\nMonitoring fine-tuning progress...")
print("This may take several minutes to hours depending on dataset size...")

while True:
    job_status = openai.fine_tuning.jobs.retrieve(job_id)
    status = job_status.status
    
    print(f"Status: {status}")
    
    if status == "succeeded":
        fine_tuned_model = job_status.fine_tuned_model
        print(f"\n✓ Fine-tuning completed!")
        print(f"Fine-tuned model ID: {fine_tuned_model}")
        break
    elif status == "failed":
        print(f"\n✗ Fine-tuning failed!")
        print(f"Error: {job_status.error}")
        break
    
    time.sleep(60)  # Check every 60 seconds

> Total time taken: 22 minutes

## Step 5: Use your fine-tuned model

### Direct use after fine-tuning

In [7]:
if status == "succeeded":
    print("\n--- Testing Fine-Tuned Model ---")
    completion = openai.chat.completions.create(
        model=fine_tuned_model,
        messages=[
            {"role": "system", "content": "You are a customer support assistant for TechCorp."},
            {"role": "user", "content": "What are your business hours?"}
        ]
    )
    
    print("Response:", completion.choices[0].message.content)


--- Testing Fine-Tuned Model ---
Response: TechCorp support is available Monday to Friday, 8 AM to 6 PM EST. We offer extended hours for premium customers.


### Load fine-tuned model from OpenAI

In [ ]:
# List all your fine-tuned models
models = openai.models.list()
fine_tuned = [m for m in models.data if m.id.startswith('ft:')]

for model in fine_tuned:
    print(f"Model: {model.id}")
    print(f"Created: {model.created}")

In [ ]:
import openai

# Just use the fine-tuned model ID directly
fine_tuned_model_id = os.getenv("OPENAI_FINE_TUNED_MODEL_ID")  # You get this after fine-tuning

# Use it in any chat completion
completion = openai.chat.completions.create(
    model=fine_tuned_model_id,  # Your fine-tuned model
    messages=[
        {"role": "system", "content": "You are a customer support assistant for TechCorp."},
        {"role": "user", "content": "How do I reset my password?"}
    ]
)

print(completion.choices[0].message.content)

To reset your password: 1) Go to login page 2) Click 'Forgot Password' 3) Enter your email 4) Check your inbox for reset link


In [ ]:
# Additional useful commands:

# List all your fine-tuning jobs
# jobs = openai.fine_tuning.jobs.list()

# Cancel a running job
# openai.fine_tuning.jobs.cancel(job_id)

# Delete a fine-tuned model
# openai.models.delete(fine_tuned_model)